# *Perkinsus marinus* infections
We're wondering if there is evidence of any low lying *P. marinus* infections in these oysters. To do this, I'm going to map my sequences to the [*P. marinus* genome](https://www.ncbi.nlm.nih.gov/datasets/genome/GCF_000006405.1/)

A lot of this code will be run through bash jobs

## 1. get *P. marinus* genome

In [ ]:
# get the genome
datasets download genome accession GCF_000006405.1 --include genome

the genome path is now: `/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/perkinsus_genome/ncbi_dataset/data/GCF_000006405.1/GCF_000006405.1_JCVI_PMG_1.0_genomic.fna`

## 2. align oyster RNA-sequences to *P. marinus* genome
submit as a job

create botwie index

In [ ]:
#!/bin/bash
#SBATCH --job-name=pmar_index
#SBATCH --cpus-per-task=16
#SBATCH --mem=32G
#SBATCH -p cpu
#SBATCH -t 1:00:00
#SBATCH -o pmar_index_%j.log
#SBATCH --mail-type=END,FAIL

module load conda/latest
conda activate samtools-env
module load bowtie2/2.5.2

PMAR="/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/perkinsus_genome/ncbi_dataset/data/GCF_000006405.1/GCF_000006405.1_JCVI_PMG_1.0_genomic.fna"

REFERENCE_INDEX="/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/pmar_reference_index"

set -euo pipefail

mkdir -p "$(dirname "${REFERENCE_INDEX}")"

bowtie2-build "${PMAR}" "${REFERENCE_INDEX}"

align seqs to indexed *P. marinus* genome

In [ ]:
#!/bin/bash
#SBATCH --job-name=pmarinus_align
#SBATCH --cpus-per-task=16
#SBATCH --mem=64G
#SBATCH -p cpu
#SBATCH -t 1:00:00
#SBATCH --array=1-120%5 # 120 array, run 5 samples at a time max
#SBATCH -o pmarinus_alignment_%A_%a.log
#SBATCH --mail-type=END,FAIL

module load conda/latest
conda activate samtools-env
module load bowtie2/2.5.2

set -euo pipefail

INPUT="/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/trimmed_all"
OUTPUT="/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/pmarinus_alignment"
REFERENCE_INDEX="/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/pmar_reference_index/pmar_reference_index"

mkdir -p "${OUTPUT}"

# Get the R1 file corresponding to this array task
R1=$(find "${INPUT}" -maxdepth 1 -name '*_gi_1_val_1.fq.gz' | sort | sed -n "${SLURM_ARRAY_TASK_ID}p")

if [[ -z "${R1}" ]]; then
    echo "ERROR: No R1 file found for array task ${SLURM_ARRAY_TASK_ID}"
    exit 1
fi

SAMPLE=$(basename "${R1}" _gi_1_val_1.fq.gz)

R2="${INPUT}/${SAMPLE}_gi_2_val_2.fq.gz"

if [[ ! -f "${R2}" ]]; then
    echo "ERROR: R2 file not found: ${R2}"
    exit 1
fi

echo "======================================"
echo "Array task: ${SLURM_ARRAY_TASK_ID}"
echo "Sample: ${SAMPLE}"
echo "R1: ${R1}"
echo "R2: ${R2}"
echo "======================================"

bowtie2 \
    --very-sensitive-local \
    --threads "${SLURM_CPUS_PER_TASK}" \
    -x "${REFERENCE_INDEX}" \
    -1 "${R1}" \
    -2 "${R2}" \
    2> "${OUTPUT}/${SAMPLE}_bowtie2.log" \
    | samtools view -@ "${SLURM_CPUS_PER_TASK}" -b \
    | samtools sort \
        -@ "${SLURM_CPUS_PER_TASK}" \
        -o "${OUTPUT}/${SAMPLE}_alignment.sorted.bam"

samtools index \
    -@ "${SLURM_CPUS_PER_TASK}" \
    "${OUTPUT}/${SAMPLE}_alignment.sorted.bam"

echo "Finished ${SAMPLE}"


I just realized that I've actually done this before when I ran fastqscreen to try and figure out what other hits I'm getting in my RNA sequences

the [multiqc results](https://github.com/jgmcdonough/CE24_RNA-seq/blob/main/processing/qc_outputs/multiqc_fastqScreen_report.html) show that 187 out of 240 sequence files show alignments to the *P. marinus* genome (this is not the number of unique samples - these are paired end reads). 

120 unique samples had sequences that aligned to the *P. marinus* genome (regardless if this was seen in both reads or just one of the pairs) - however, for all of these, less than 0.01% of the reads aligned to the *P. marinus* genome (but this is probably still thousands of reads)